In [ ]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input LF and HF interpolated backgroud/mission path data CSVs
lf_csv = r"C:\Users\kkeii\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\LF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"
hf_csv = r"C:\Users\kkeii\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\HF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"


# intermediate polyline feature classes output by 'Coordinate Table to 2-Point Line' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_line_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_line_g4326"
hf_line_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_line_g4326"
# intermediate point feature classes output by 'XY Table to Point' tool before reprojection (*output to misc_around_stx_2025.gdb)
lf_points_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\LF_background_points_g4326"
hf_points_g4326 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\misc_around_stx_2025.gdb\HF_background_points_g4326"


# Output MISSION PATH LINE feature classes (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_line_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\LF_background_line_p26920"
hf_line_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\HF_background_line_p26920"

# Output CALC ATTRIBUTE TABLE point feature classes (reprojected to EPSG:26920, *output to GIS_UVI2_Around_STX_20250306.gdb)
lf_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_LF_background_points_p26920"
hf_points_p26920 = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Missions\GIS_UVI2_Around_STX_20250306\GIS_UVI2_Around_STX_20250306.gdb\calc_HF_background_points_p26920"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── 1. CREATE MISSION PATH LINE FROM CSV DATA ────────────────────────────────────────────────────────

# Coordinate Table To 2-Point Line (Defense Tools): Function
def create_mission_path_line(in_csv, out_line_g4326):
    
    try:
        arcpy.defense.CoordinateTableTo2PointLine(
            in_table=in_csv,
            out_feature_class=out_line_g4326,
            start_x_or_lon_field="start_longitude_DD",
            end_x_or_lon_field="end_longitude_DD",
            # input CSV has coordinates in decimal degrees (DD) with lat and lon stored in * two separate columns [lat] [lon] * (not in one column separated by comma [lat, lon])
            in_coordinate_format="DD_2",
            start_y_or_lat_field="start_latitude_DD",
            end_y_or_lat_field="end_latitude_DD",
            line_type="GEODESIC",
            # output line feature class in EPSG:4326 (native coordinate system of glider data that coordinates in CSVs are interpolated from)
            coordinate_system=arcpy.SpatialReference(4326)
        )
        print(f"✅ Created 2-point line feature class: {out_line_g4326} \n")
    
    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] 2-point line feature class creation failed for {os.path.basename(out_line_g4326)}: {e} \n")


# Call function for LF and HF mission background conditions CSVs
create_mission_path_line(lf_csv, lf_line_g4326)
create_mission_path_line(hf_csv, hf_line_g4326)

In [ ]:
# ── 2. CREATE POINT FEATURE CLASSES FROM CSV DATA ────────────────────────────────────────────────────

# XY Table To Point (Data Management Tools): Function
def xy_table_to_point(in_csv, out_points_g4326):
    
    try:
        arcpy.management.XYTableToPoint(
            in_table=in_csv,
            out_feature_class=out_points_g4326,
            # use lat/lon at * middle-point * of each spectrogram minute 
            x_field="av_longitude_DD",
            y_field="av_latitude_DD",
            z_field=None,
            # output point feature class in EPSG:4326 (native coordinate system of glider data that coordinates in CSVs are interpolated from)
            coordinate_system=arcpy.SpatialReference(4326)
        )
        print(f"✅ Created point feature class: {out_points_g4326} \n")
        
    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] Point feature class creation failed for {os.path.basename(out_points_g4326)}: {e} \n")


# Call function for LF and HF mission background conditions CSVs
xy_table_to_point(lf_csv, lf_points_g4326)
xy_table_to_point(hf_csv, hf_points_g4326)

In [ ]:
# ── 3. REPROJECT FEATURE CLASSES CREATED FROM CSV DATA TO EPSG:26920 ─────────────────────────────────

# Project (Data Management Tools): Function
def reproject_fc(in_fc, out_fc):
    
    try:
        arcpy.management.Project(
            in_dataset=in_fc,
            out_dataset=out_fc,
            # output coordinate system
            out_coor_system=arcpy.SpatialReference(26920),
            transform_method="WGS_1984_(ITRF00)_To_NAD_1983",
            # input coordinate system
            in_coor_system=arcpy.SpatialReference(4326),
            # * do NOT want tool to add extra vertices to feature classes (want to preserve input data)
            preserve_shape="NO_PRESERVE_SHAPE"
        )
        print(f"✅ Reprojected feature class: {out_fc} \n")

    # print warning message if tool fails
    except Exception as e:
        print(f"❌[FAILURE] Reprojection failed for {os.path.basename(out_fc)}: {e} \n")


# Call function for LF and HF mission path line feature classes + background conditions point feature classes
reproject_fc(lf_line_g4326, lf_line_p26920)
reproject_fc(hf_line_g4326, hf_line_p26920)
reproject_fc(lf_points_g4326, lf_points_p26920)
reproject_fc(hf_points_g4326, hf_points_p26920)

In [ ]:
# ── INSPECT REPROJECTED LINE AND POINT FEATURE CLASSES ───────────────────────────────────────────────

# Function to print info (geometry type, feature count, coordinate systems) for each feature class
def print_fc_info(fc):

    # get geometry type/coordinate system info
    d = arcpy.Describe(fc)
    # get feature count info
    count = arcpy.management.GetCount(fc)[0]
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for feature class
    print(f"\n---------------- {os.path.basename(fc)} ----------------: \n")
    print(f"   Geometry type: {d.shapeType}")
    print(f"   Feature count: {count}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Call function to print info for each output reprojected line/point feature class
print_fc_info(lf_line_p26920)
print_fc_info(hf_line_p26920)
print_fc_info(lf_points_p26920)
print_fc_info(hf_points_p26920)